# Molecular Devices FilterMax F5

```{device-card} molecular-devices-filtermax-f5
```

| Property | Value |
|---|---|
| PLR class | `FilterMaxF5` |
| Connection | RS-232 serial, automatic 38,400/9,600 baud detection, 7E1, no flow control |
| Current verified modes | Absorbance and luminescence |
| Current verified controls | Tray, filter slides, temperature, shaking, cancellation |

> **Validation status:** Direct PLR-to-F5 hardware validation succeeded. Full-plate absorbance at 450 nm agreed with the corresponding SoftMax Pro results; additional validated operations are recorded in the [protocol coverage](protocol-coverage.md). Fluorescence, FP, and TRF commands remain unavailable until they can be captured with a compatible excitation slide; the driver does not infer them.

## How it communicates

The F5 uses an ASTM-like framed serial protocol, not the SpectraMax `!COMMAND` protocol. PLR handles framing, checksums, acknowledgements, response chunking, device errors, and cancellation internally. Raw commands are intentionally not public.

## Physical setup

1. Connect the FilterMax F5 through its RS-232 cable or a USB-to-serial adapter.
2. Close SoftMax Pro and release the serial port. If Windows runs in VirtualBox with a host serial port mapping, save the results and shut Windows down normally; confirm the VM is powered off. Only one program may own the port.
3. Export your filter-slide definitions from SoftMax Pro as XML, or use the example catalog below when its definitions match your physical slides.
4. Install the plate and filter slides appropriate for the requested measurement.
5. Use a stable device path on Linux, such as `/dev/serial/by-id/...`.

Excitation and emission slides occupy separate bays and are not interchangeable.

## Load the filter-slide catalog

Download {download}`filtermax_example_slides.xml <filtermax_example_slides.xml>` or use the copy alongside this notebook in the repository. It contains 16 filter definitions for excitation slides 1 and 2 and emission slide 1. Excitation slide 2 includes 260 nm at position 1 (12 nm bandwidth) and 450 nm at position 4 (8 nm bandwidth).

`FilterSlideCatalog.from_softmax_xml()` loads slide IDs, positions, wavelengths, bandwidths, and supported filter techniques. Measurement methods use this catalog to resolve a wavelength to a slide and position and to send that slide's filter definitions to the reader. Loading or resolving the catalog is local and does not contact the instrument.

Use your own SoftMax XML export for a different filter configuration. The driver checks installed slide IDs, but it cannot verify that the physical filters match the XML. Catalog entries for fluorescence, FP, and TRF do not enable those currently unsupported driver modes.

The next cell assumes this notebook's directory is the working directory. From the repository root, set `catalog_path` to `Path("docs/user_guide/molecular_devices/filtermax/filtermax_example_slides.xml")`. For your own export, use its local path instead.

In [1]:
from pathlib import Path

from pylabrobot.molecular_devices.filtermax import FilterMaxF5, FilterSlideCatalog

catalog_path = Path("filtermax_example_slides.xml")
catalog = FilterSlideCatalog.from_softmax_xml(catalog_path)

## Check an example filter locally

Resolve the 450 nm absorbance filter to excitation slide 2, position 4. For 260 nm, use `wavelength=260` and `position=1`. This check does not move the reader or start a measurement.

In [2]:
filter_selection = catalog.resolve(
    kind="excitation",
    technique="absorbance",
    wavelength=450,
    slide_id=2,
    position=4,
)
filter_selection

FilterSelection(kind='excitation', slide_id=2, position=4, wavelength=450, bandwidth=8)

## Choose the serial port

List detected serial ports and identify the FilterMax F5 adapter. On Linux, prefer its stable `/dev/serial/by-id/...` path; Windows ports usually look like `COM3`. If no port is listed, connect the adapter and check its cable/permissions. Use the exact device path shown for the F5 below.

In [3]:
from serial.tools import list_ports

[(port.device, port.description) for port in list_ports.comports()]

[('/dev/ttyS31', 'n/a'),
 ('/dev/ttyS30', 'n/a'),
 ('/dev/ttyS29', 'n/a'),
 ('/dev/ttyS28', 'n/a'),
 ('/dev/ttyS27', 'n/a'),
 ('/dev/ttyS26', 'n/a'),
 ('/dev/ttyS25', 'n/a'),
 ('/dev/ttyS24', 'n/a'),
 ('/dev/ttyS23', 'n/a'),
 ('/dev/ttyS22', 'n/a'),
 ('/dev/ttyS21', 'n/a'),
 ('/dev/ttyS20', 'n/a'),
 ('/dev/ttyS19', 'n/a'),
 ('/dev/ttyS18', 'n/a'),
 ('/dev/ttyS17', 'n/a'),
 ('/dev/ttyS16', 'n/a'),
 ('/dev/ttyS15', 'n/a'),
 ('/dev/ttyS14', 'n/a'),
 ('/dev/ttyS13', 'n/a'),
 ('/dev/ttyS12', 'n/a'),
 ('/dev/ttyS11', 'n/a'),
 ('/dev/ttyS10', 'n/a'),
 ('/dev/ttyS9', 'n/a'),
 ('/dev/ttyS8', 'n/a'),
 ('/dev/ttyS7', 'n/a'),
 ('/dev/ttyS6', 'n/a'),
 ('/dev/ttyS5', 'n/a'),
 ('/dev/ttyS4', 'n/a'),
 ('/dev/ttyS3', 'n/a'),
 ('/dev/ttyS2', 'n/a'),
 ('/dev/ttyS1', 'n/a'),
 ('/dev/ttyS0', 'n/a')]

## Create the reader

Pass the loaded catalog as `filter_slide_catalog`. Creating the reader does not open the serial port. Connection and status queries can run without a catalog, but measurement methods require one. Enter the exact FilterMax F5 port from the list above when prompted.

In [4]:
port = input("FilterMax serial port: ").strip()
if not port:
    raise ValueError("Enter the serial port for the FilterMax F5.")

reader = FilterMaxF5(
    name="filtermax-f5",
    port=port,
    filter_slide_catalog=catalog,
)

ValueError: Enter the serial port for the FilterMax F5.

## Connect

`setup()` tries 38,400 baud first, then closes and reopens the port at 9,600 baud only if the initial handshake receives no response. It verifies the F5 identity and retains the working baud rate, which appears in the connection log. It does not change the reader's baud setting. Identity, protocol, and later transaction errors stop setup without retrying. All failed setup attempts release the serial port.

In [ ]:
await reader.setup()

## Inspect the instrument

Read identity, status, installed slide IDs, and temperature without changing instrument state.

In [ ]:
info = await reader.get_instrument_info()
status = await reader.get_status()
slides = await reader.get_filter_slides()
temperature = await reader.get_temperature()

info, status, slides, temperature

## Plate tray

Move the tray out before loading or removing a plate.

In [ ]:
await reader.move_plate_tray_out()

After the plate is loaded and the tray path is clear, move it back in.

In [ ]:
await reader.move_plate_tray_in()

## Temperature control

Set a temperature in degrees Celsius, then query the current chamber temperature.

In [ ]:
await reader.set_temperature(25.0)
temperature = await reader.get_temperature()

Deactivate temperature control when it is no longer needed.

In [ ]:
await reader.deactivate_temperature_control()

## Shaking

Shaking is a finite-duration operation. Patterns are `"linear"` or `"orbital"`; speeds are `"low"`, `"medium"`, or `"high"`.

In [ ]:
await reader.start_shaking(pattern="linear", speed="medium", duration=5)

## Plate geometry

The captured Costar 96-well clear plate geometry is available as a convenience. For another plate, construct `PlateGeometry` using its measured dimensions in millimetres.

In [ ]:
from pylabrobot.molecular_devices.filtermax import PlateGeometry

plate = PlateGeometry.costar_96_clear_landscape()

## Absorbance

Run an endpoint read at 450 nm. Results are plate-shaped optical-density values; unselected wells are `None`. The tray ejects when the read finishes.

In [ ]:
absorbance = await reader.read_absorbance(plate, wavelength=450)

Select individual wells for a partial-plate read.

In [ ]:
partial = await reader.read_absorbance(
    plate,
    wavelength=450,
    wells=["C4", "C5", "C6", "D4", "D5", "D6"],
)

Use a supported reference filter when reference-subtracted absorbance is required.

In [ ]:
reference_subtracted = await reader.read_absorbance(
    plate,
    wavelength=450,
    reference_wavelength=620,
    wells=["C4", "C5", "C6", "D4", "D5", "D6"],
)

## Kinetic reads

Kinetic timing uses seconds. The result list contains one plate-shaped result per timepoint.

In [ ]:
from pylabrobot.molecular_devices.filtermax import KineticTiming

kinetic = await reader.read_absorbance(
    plate,
    wavelength=450,
    wells=["C4"],
    kinetic=KineticTiming(interval=15, reads=3),
)

## Well scans

The captured F5 software exposes horizontal and circular fill scans. Scan results preserve every `(x, y, value)` point for each selected well.

In [ ]:
from pylabrobot.molecular_devices.filtermax import WellScanSettings

scan = await reader.read_absorbance(
    plate,
    wavelength=450,
    wells=["C4"],
    well_scan=WellScanSettings(pattern="fill", density=5),
)

## Luminescence

Read the open luminescence position on the installed emission slide. Values are raw RLU. Set `channels=2` for the captured dual-luminescence operation.

In [ ]:
luminescence = await reader.read_luminescence(
    plate,
    wells=["C4"],
    integration=0.4,
    read_height=1.0,
)

## Cancel a read

`cancel_read()` is state-aware: it does nothing while idle and cooperatively stops an active read. A cancelled read raises `FilterMaxReadCancelled` in the read task and returns the reader to an idle, tray-ejected state.

In [ ]:
import asyncio

from pylabrobot.molecular_devices.filtermax import FilterMaxReadCancelled

read_task = asyncio.create_task(
    reader.read_absorbance(
        plate,
        wavelength=450,
        wells=["C4"],
        kinetic=KineticTiming(interval=30, reads=10),
    )
)
await asyncio.sleep(2)
await reader.cancel_read()
try:
    await read_task
except FilterMaxReadCancelled:
    pass

## Device errors

Inspect the exact firmware errors observed during this PLR session, then clear the local session log when handled.

In [ ]:
errors = await reader.get_error_log()
await reader.clear_error_log()

## Disconnect

Always release the serial port when finished.

In [ ]:
await reader.stop()